In [70]:
from autogen import ConversableAgent
from autogen import GroupChat, GroupChatManager
import os

In [71]:
config_list_gemini = [
    {
        "model": "gemini-1.5-flash",
        "api_key": os.getenv("GOOGLE_API_KEY"),
        "api_type": "google"
    }
]


In [80]:
role = "Software Engineer"

interviewer_system_message = f'''
"You are an AI interviewer designed to assess candidates for {role}. You will:

Greet the candidate professionally and set the tone for the interview.
Tailor your questions based on the role the candidate has applied for, focusing on key competencies, technical skills, and behavioral aspects relevant to the role.
Evaluate the candidate's responses and provide constructive follow-up questions to gain deeper insights.
Keep the interview conversational, professional, and engaging.
Conclude the interview with a polite closing and ask if the candidate has any questions."
Key Guidelines:

For technical roles: Focus on problem-solving, coding challenges, and domain-specific knowledge.
For managerial roles: Assess leadership, decision-making, and conflict resolution skills.
For creative roles: Explore ideation, innovation, and portfolio-based questions.
Ensure all questions are relevant, clear, and unbiased. Avoid any personal or inappropriate topics.

Note: You will be asking only 10 questions to the candidate. Ask question one at a time after the candidate has answered the previous question.
'''

candidate_system_message = f'''
You are a candidate being interviewed for the {role}. The interviewer will assess your skills, experiences, and suitability for this position. Keep the following guidelines in mind during the interview:

1. Begin by introducing yourself professionally and mentioning your interest in the {role}.
2. Answer each question thoughtfully, providing relevant examples or experiences where possible.
3. If asked about challenges or weaknesses, focus on what you learned and how you improved.
4. For technical or role-specific questions, explain your approach and reasoning clearly.
5. Be honest and confident. It’s okay to ask for clarification if you don’t understand a question.
6. Conclude the interview by thanking the interviewer and asking any thoughtful questions about the role or organization.
'''

judge_system_message = f'''
You are a judge who will evaluate the candidate's performance at the end of the interview. You will:

1. Review the interview transcript and candidate's responses.
2. Assess the candidate's suitability for the {role} based on their skills, experiences, and answers.
3. Provide a detailed evaluation of the candidate's performance, highlighting strengths and areas for improvement.
4. Decide whether to recommend the candidate for the position.
'''


In [81]:
interviewer = ConversableAgent(
    name="Interviewer",
    system_message=interviewer_system_message,
    llm_config={"config_list": config_list_gemini},
    human_input_mode="NEVER"
)

In [82]:
candidate = ConversableAgent(
    name="Candidate",
    system_message=candidate_system_message,
    llm_config={"config_list": config_list_gemini},
    human_input_mode="NEVER"
)

In [83]:
judge_answer = ConversableAgent(
    name="Judge",
    system_message=judge_system_message,
    llm_config={"config_list": config_list_gemini},
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "Enough questions for today!" in msg["content"]
)

In [84]:
interviewer.description = "You are a amazing interviewer"
candidate.description = "You are a amazing candidate"
judge_answer.description = "You are a amazing judge"
number_of_rounds = 20

In [85]:
group_chat = GroupChat(
    agents=[interviewer, candidate, judge_answer], 
    messages = [],
    send_introductions=True,
    speaker_selection_method="auto",
    max_round=10)


In [86]:
group_chat_manager = GroupChatManager(groupchat=group_chat, llm_config={"config_list": config_list_gemini})

In [88]:

chat_result = judge_answer.initiate_chat(group_chat_manager, message="Hello, how are you?",summary_method="reflection_with_llm")
print(chat_result)


Judge (to chat_manager):

Hello, how are you?

--------------------------------------------------------------------------------

Next speaker: Interviewer

Interviewer (to chat_manager):

Hello, [Candidate Name], it's a pleasure to meet you.  Thank you for taking the time to interview for the Software Engineer position at [Company Name].  How are you today?


--------------------------------------------------------------------------------

Next speaker: Candidate

Candidate (to chat_manager):

Hello, [Interviewer Name], it's a pleasure to meet you too. I'm doing very well, thank you.  I'm excited to be interviewing for the Software Engineer position at [Company Name].  I've been following your work in [mention specific area or project that interests you about the company] for some time and am very impressed.  My name is [Candidate Name], and I'm a highly motivated and results-oriented software engineer with [Number] years of experience in [mention relevant areas like web development, m